# 🗳️ [Super AI Engineer Season 6] การแข่งขัน OCR เอกสารผลเลือกตั้ง สส. 2569

แนวคิดหลักคือเลิกใช้ OCR แบบเดิมๆ (Text detection -> Recognition) แล้วเปลี่ยนมาใช้ **Visual Understanding** แทนครับ โดยใช้โมเดล **Gemini 2.0 Flash** ผ่าน OpenRouter ซึ่งเก่งมากเรื่องการอ่านตารางภาษาไทยที่ซับซ้อน

In [ ]:
import os
import subprocess
import sys

def run(cmd):
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

# Install OpenAI SDK for OpenRouter & Dependencies
# typhoon_ocr is optional (backend-dependent), but pandas/dotenv/tqdm are core.
run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 
     'openai', 'pandas', 'tqdm', 'python-dotenv', 'typhoon_ocr'])

In [ ]:
# Auto-detect dataset path to fix FileNotFoundError
import os

possible_roots = [
    '/kaggle/input/super-ai-engineer-season-6-ocr-2569/data',      # Standard Kaggle
    '/kaggle/input/super-ai-engineer-season-6-ocr-2569',           # Alternative Kaggle
    './data/data',                                                 # Local unzip style 1
    './data',                                                      # Local unzip style 2
    '.'                                                            # Current dir
]

TEMPLATE_CSV = None
BASE_DIR = ''

for root in possible_roots:
    path = os.path.join(root, 'submission_template.csv')
    if os.path.exists(path):
        TEMPLATE_CSV = path
        BASE_DIR = root
        break

if TEMPLATE_CSV is None:
    # Fallback default if nothing found
    BASE_DIR = '/kaggle/input/super-ai-engineer-season-6-ocr-2569/data'
    TEMPLATE_CSV = os.path.join(BASE_DIR, 'submission_template.csv')
    print(f"Warning: Dataset not found. Defaulting to: {TEMPLATE_CSV}")

IMAGE_DIR = os.path.join(BASE_DIR, 'images')
SAMPLE_LABEL_DIR = os.path.join(BASE_DIR, 'sample_labels')

# Working paths
PIPELINE_SCRIPT = 'embedded_pipeline.py'
OUTPUT_CSV = 'submission.csv'
SAMPLE_GOLD_CSV = 'sample_gold.csv'

print(f"Template path: {TEMPLATE_CSV}")
print(f"Image dir:     {IMAGE_DIR}")
print(f"Label dir:     {SAMPLE_LABEL_DIR}")


In [ ]:
import base64
import json
import os
import re
import threading
import time
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from difflib import get_close_matches
from pathlib import Path
from typing import Protocol, Dict, List, Optional

import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm

# --- 1. CONFIGURATION ---
API_KEY = os.getenv("OPENROUTER_API_KEY", "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")
if not API_KEY:
    raise ValueError("Missing API Key. Please set the OPENROUTER_API_KEY environment variable.")

MAX_WORKERS = os.cpu_count() or 4
CACHE_DIR = Path("./cache_sota_v25")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

OCR_CACHE_CSV = CACHE_DIR / "ocr_cache.csv"
LLM_CACHE_CSV = CACHE_DIR / "llm_cache.csv"

# --- 2. CORE CLASSES ---

class RateLimiter:
    """Sliding-window rate limiter supporting multiple simultaneous constraints."""
    def __init__(self, limits: list[tuple[int, float]]):
        self._limits = limits
        self._windows: list[deque] = [deque() for _ in limits]
        self._lock = threading.Lock()

    def acquire(self) -> None:
        while True:
            with self._lock:
                now = time.monotonic()
                for (max_req, window), dq in zip(self._limits, self._windows):
                    while dq and now - dq[0] >= window:
                        dq.popleft()
                if all(len(dq) < max_req for (max_req, _), dq in zip(self._limits, self._windows)):
                    for dq in self._windows:
                        dq.append(now)
                    return
                wait = min((dq[0] + window - now) for (max_req, window), dq in zip(self._limits, self._windows) if len(dq) >= max_req)
                time.sleep(max(wait, 0.05))

class OcrBackend(Protocol):
    def __call__(self, image_path: Path) -> str: ...

_VLM_OCR_PROMPT = (
    "This is a page from a Thai official election result form (สส.6/1). "
    "Your task is to perform a complete, verbatim transcription of every piece of information on this page. "
    "Output only Markdown — no commentary, no explanations, nothing outside the transcription.\n\n"
    "Follow these rules strictly:\n"
    "1. **Transcribe every row** of every table without exception. Never skip, truncate, or summarise rows.\n"
    "2. **Preserve exact Thai text** for all party names (ชื่อพรรค), candidate names (ชื่อ-สกุล), "
    "   section labels, and header fields exactly as printed.\n"
    "3. **Convert Thai numerals** (๐๑๒๓๔๕๖๗๘๙) to Arabic digits (0–9) in all numeric fields.\n"
    "4. **For vote-count tables**, reproduce each data row as a Markdown table row with columns:\n"
    "   | หมายเลข | ชื่อ-สกุล | สังกัดพรรคการเมือง | คะแนน |\n"
    "   Include the header row and a separator row.\n"
    "5. **For summary statistics** (numbered items such as 1.1, 1.2, 2.1, 3.1 …), transcribe each item "
    "   on its own line in the format: `<item number> <label> <value>`.\n"
    "6. **Transcribe all header fields** (province, constituency/district name, polling station, date, "
    "   official signatures section labels) verbatim.\n"
    "7. If a cell is blank or illegible, write `-`.\n"
    "8. Do **not** add any text, commentary, or formatting that is not present in the image."
)

ocr_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY)

def _ocr_openrouter_vlm(image_path: Path, model: str) -> str:
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    resp = ocr_client.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": _VLM_OCR_PROMPT},
            ],
        }],
        temperature=0,
    )
    return resp.choices[0].message.content or ""

# --- BACKEND REGISTRY ---
OCR_BACKENDS: dict[str, OcrBackend] = {
    "gemini-2.0-flash": lambda p: _ocr_openrouter_vlm(p, model="google/gemini-2.0-flash-001"),
    "gemini-2.5-flash": lambda p: _ocr_openrouter_vlm(p, model="gemini-2.5-flash"),
}

OCR_BACKEND_NAME = "gemini-2.5-flash"

OCR_RATE_LIMITS = {
    "gemini-2.0-flash": [(10, 1.0), (50, 60.0)],
    "gemini-2.5-flash": [(15, 1.0), (75, 60.0)],
}

ocr_backend = OCR_BACKENDS[OCR_BACKEND_NAME]
ocr_rate_limiter = RateLimiter(limits=OCR_RATE_LIMITS.get(OCR_BACKEND_NAME, [(5, 1.0)]))

print(f"Setup complete. OCR backend: {OCR_BACKEND_NAME!r}")

In [ ]:
# --- 3. DATASET GROUPING (Matching solution.ipynb) ---

def get_page_order(name: str) -> int:
    """constituency_10_1.png → 1, constituency_10_1_page2.png → 2"""
    m = re.search(r"_page(\d+)\.png$", name)
    return int(m.group(1)) if m else 1


def get_doc_id(name: str) -> str:
    """Strip _pageN.png suffix to get doc_id."""
    return re.sub(r"_page\d+\.png$", "", name).removesuffix(".png")

# Use IMAGE_DIR defined in Cell 3
doc_pages: dict[str, list[Path]] = defaultdict(list)
for img in Path(IMAGE_DIR).glob("*.png"):
    doc_pages[get_doc_id(img.name)].append(img)

for doc_id in doc_pages:
    doc_pages[doc_id].sort(key=lambda p: get_page_order(p.name))

# Use TEMPLATE_CSV defined in Cell 3
template = pd.read_csv(Path(TEMPLATE_CSV), dtype=str)
template = template.dropna(subset=["doc_id", "party_name"])
template = template[template["party_name"].str.strip() != ""]

doc_parties: dict[str, list[str]] = (
    template.groupby("doc_id")["party_name"].apply(list).to_dict()
)

print(f"Documents : {len(doc_pages)}")
print(f"Pages     : {sum(len(v) for v in doc_pages.values())}")
print(f"Template  : {len(template)} rows across {len(doc_parties)} docs")

In [ ]:
# --- 4. BATCH OCR (Matching solution.ipynb) ---

_ocr_write_lock = threading.Lock()

def _load_ocr_cache() -> dict[tuple[str, int], str]:
    if OCR_CACHE_CSV.exists():
        df = pd.read_csv(OCR_CACHE_CSV, dtype=str)
        # Ensure page is int
        df["page"] = pd.to_numeric(df["page"], errors='coerce').fillna(1).astype(int)
        df["ocr_text"] = df["ocr_text"].fillna("")
        return {(row.doc_id, int(row.page)): row.ocr_text for row in df.itertuples()}
    return {}


def _save_ocr_cache(cache: dict[tuple[str, int], str]) -> None:
    rows = [
        {"doc_id": doc_id, "page": page, "ocr_text": text}
        for (doc_id, page), text in cache.items()
    ]
    df = pd.DataFrame(rows, columns=["doc_id", "page", "ocr_text"])
    df.sort_values(["doc_id", "page"]).reset_index(drop=True).to_csv(OCR_CACHE_CSV, index=False)


def _ocr_one_page(doc_id: str, page_num: int, page: Path) -> tuple[str, int, Optional[str]]:
    with _ocr_write_lock:
        cached = ocr_cache.get((doc_id, page_num))
        if cached is not None and cached.strip():
            return doc_id, page_num, None  # already good

    ocr_rate_limiter.acquire()
    for attempt in range(3):
        try:
            # Use the global ocr_backend function
            text = ocr_backend(page)
            if text and text.strip():
                return doc_id, page_num, text
            print(f"  OCR returned blank [{doc_id} p{page_num}] attempt {attempt + 1}")
            time.sleep(2 ** attempt)
        except Exception as e:
            print(f"  OCR attempt {attempt + 1} failed [{doc_id} p{page_num}]: {e}")
            time.sleep(2 ** attempt)
    return doc_id, page_num, ""

# Load Cache
ocr_cache = _load_ocr_cache()
print(f"OCR cache loaded: {len(ocr_cache)} pages.")

work: list[tuple[str, int, Path]] = []
for doc_id, pages in doc_pages.items():
    for page_num, page in enumerate(pages, start=1):
        key = (doc_id, page_num)
        if key not in ocr_cache or not ocr_cache[key].strip():
            work.append((doc_id, page_num, page))

print(f"Pages to OCR: {len(work)}")

if work:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(_ocr_one_page, doc_id, page_num, page): (doc_id, page_num)
            for doc_id, page_num, page in work
        }
        with tqdm(total=len(work), desc=f"Batch OCR [{OCR_BACKEND_NAME}]") as pbar:
            for future in as_completed(futures):
                d_id, pg_num, text = future.result()
                if text is not None:
                    with _ocr_write_lock:
                        ocr_cache[(d_id, pg_num)] = text
                        # Periodically save, or save strictly every time
                        _save_ocr_cache(ocr_cache)
                pbar.update(1)

print("OCR Loop Finished.")

# Prepare doc_ocr dict for next step
doc_ocr: dict[str, str] = {
    doc_id: "\n\n---PAGE BREAK---\n\n".join(
        ocr_cache.get((doc_id, page_num), "")
        for page_num, _ in enumerate(pages, start=1)
    )
    for doc_id, pages in doc_pages.items()
}

In [ ]:
# --- 6. SUBMISSION GENERATION (Matching solution.ipynb) ---

def match_to_template(extracted: dict[str, int], parties: list[str], doc_id: str = "") -> dict[str, int]:
    result = {p: 0 for p in parties}
    unmatched = []
    for llm_name, votes in extracted.items():
        if not isinstance(llm_name, str) or not llm_name.strip():
            continue
        
        # Exact match
        if llm_name in result:
            result[llm_name] = votes
        else:
            # Fuzzy match
            close = get_close_matches(llm_name, parties, n=1, cutoff=0.7)
            if close:
                result[close[0]] = votes
            else:
                unmatched.append(llm_name)
    
    return result

all_results: dict[str, dict[str, int]] = {}

for doc_id in doc_pages:
    parties = doc_parties.get(doc_id, [])
    if not parties:
        continue

    if doc_id in llm_cache:
        # Step 1: Match extraction to template
        all_results[doc_id] = match_to_template(llm_cache[doc_id], parties, doc_id)
    else:
        # Step 2: Fallback to all zeros if extraction missing
        all_results[doc_id] = {p: 0 for p in parties}

# Generate Submission
try:
    sample_submission = pd.read_csv(TEMPLATE_CSV, dtype=str)
    
    # Fill votes
    sample_submission["votes"] = sample_submission.apply(
        lambda row: all_results.get(row["doc_id"], {}).get(row["party_name"], 0),
        axis=1,
    ).astype(int)

    # Save
    sample_submission[["id", "votes"]].to_csv(OUTPUT_CSV, index=False)

    total_rows   = len(sample_submission)
    nonzero_rows = int((sample_submission["votes"] > 0).sum())
    print(f"Saved {OUTPUT_CSV}: {total_rows} rows")
    print(f"  non-zero votes: {nonzero_rows}")
    
    display(sample_submission[["id", "votes"]].head(20))
except Exception as e:
    print(f"Error generating submission: {e}")

In [ ]:
# --- 5. BATCH LLM EXTRACTION (Matching solution.ipynb) ---

CONSTITUENCY_PROMPT = """You are extracting vote counts from a scanned Thai election result form (สส.6/1).
This is a constituency (แบบแบ่งเขต) document — each row has a candidate name, party, and vote count.

For each party in the list, find and return its total vote count.
Return ONLY valid JSON: {{"party_name": vote_count, ...}}
All vote counts must be integers (Arabic digits 0-9). Use 0 if a party is not found.

Parties to extract:
{party_list}

OCR text:
{ocr_text}"""

PARTY_LIST_PROMPT = """You are extracting vote counts from a scanned Thai election result form (สส.6/1).
This is a party-list (แบบบัญชีรายชื่อ) document — 57 parties numbered 1-57, no candidate names.

For each party in the list, find and return its vote count.
Return ONLY valid JSON: {{"party_name": vote_count, ...}}
All vote counts must be integers (Arabic digits 0-9). Use 0 if a party is not found.

Parties to extract:
{party_list}

OCR text:
{ocr_text}"""

_llm_lock = threading.Lock()
llm_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY)
# LLM Logic Model - keeping Gemini 2.0 Flash as it is good for logic
EXTRACT_MODEL = "google/gemini-2.0-flash-001"

def _load_llm_cache() -> dict[str, dict[str, int]]:
    if not LLM_CACHE_CSV.exists():
        return {}
    df = pd.read_csv(LLM_CACHE_CSV, dtype=str)
    df = df.dropna(subset=["doc_id", "party_name", "votes"])
    df = df[df["party_name"].str.strip() != ""]
    result: dict[str, dict[str, int]] = defaultdict(dict)
    for row in df.itertuples():
        try:
            result[str(row.doc_id)][str(row.party_name)] = int(float(row.votes))
        except (ValueError, TypeError):
            pass
    return dict(result)


def _save_llm_cache(cache: dict[str, dict[str, int]]) -> None:
    rows = [
        {"doc_id": doc_id, "party_name": party, "votes": votes}
        for doc_id, parties in cache.items()
        for party, votes in parties.items()
    ]
    pd.DataFrame(rows, columns=["doc_id", "party_name", "votes"]) \
      .sort_values(["doc_id", "party_name"]).reset_index(drop=True) \
      .to_csv(LLM_CACHE_CSV, index=False)


def _extract_votes_one(doc_id: str, ocr_text: str, parties: list[str]) -> tuple[str, Optional[dict[str, int]]]:
    with _llm_lock:
        if doc_id in llm_cache:
            return doc_id, None
    
    is_party_list = "party_list" in doc_id # Simple check based on filename/id pattern
    prompt = (PARTY_LIST_PROMPT if is_party_list else CONSTITUENCY_PROMPT).format(
        party_list="\n".join(f"- {p}" for p in parties),
        ocr_text=ocr_text[:12000],
    )
    
    # Rate limit check (using same limiter or a separate one?)
    # Solution.ipynb reused ocr_rate_limiter or didn't rate limit LLM extraction explicitly in the loop 
    # but relied on max_workers. We'll add a small sleep to be safe.
    time.sleep(0.5) 

    for attempt in range(3):
        try:
            resp = llm_client.chat.completions.create(
                model=EXTRACT_MODEL,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0,
            )
            raw = json.loads(resp.choices[0].message.content)
            result: dict[str, int] = {}
            for k, v in raw.items():
                try:
                    result[str(k)] = int(float(v))
                except (TypeError, ValueError):
                    pass
            return doc_id, result
        except Exception as e:
            print(f"  LLM attempt {attempt+1} failed [{doc_id}]: {e}")
            time.sleep(2 ** attempt)
    return doc_id, {p: 0 for p in parties}

llm_cache = _load_llm_cache()
print(f"LLM cache loaded: {len(llm_cache)} docs.")

# Define work queue
work_q5 = [
    (doc_id, doc_ocr[doc_id], doc_parties.get(doc_id, []))
    for doc_id in doc_ocr
    if doc_id in doc_parties and doc_id not in llm_cache
]

print(f"Docs to extract: {len(work_q5)}")

if work_q5:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(_extract_votes_one, doc_id, ocr_text, parties): doc_id
            for doc_id, ocr_text, parties in work_q5
        }
        with tqdm(total=len(work_q5), desc="Batch LLM extraction") as pbar:
            for future in as_completed(futures):
                doc_id, result = future.result()
                if result is not None:
                    with _llm_lock:
                        llm_cache[doc_id] = result
                        _save_llm_cache(llm_cache)
                pbar.update(1)

print("Extraction Loop Finished.")